# 非评分实验：Chunking
---
欢迎来到 Chunking 非评分实验！正如课程中所介绍的，chunking 会将大文本拆分为更小、可管理的片段，这对于高效使用向量数据库和语言模型非常关键。


# 目录
- [ 1 - 介绍](#1)
  - [ 1.1 导入所需库](#1-1)
  - [ 1.2 下载数据](#1-2)
- [ 2 - 固定长度切分](#2)
  - [ 2.1 切分代码示例](#2-1)
  - [ 2.2 带重叠切分](#2-2)
- [ 3 - 可变长度切分 - 递归字符分割](#3)
  - [ 3.1 可变长度切分方法伪代码](#3-1)
  - [ 3.2 混合固定与可变长度切分](#3-2)
- [ 4 - 在真实数据上切分](#4)
  - [ 4.1 获取数据](#4-1)
  - [ 4.2 章节切分](#4-2)
  - [ 4.3 将切分结果加载到向量数据库](#4-3)
- [ 5 - 搜索 ](#5)
- [ 6 - 在 RAG 系统中集成](#6)


<a id='1'></a>
## 1 - 介绍

---

Chunking 在信息检索中非常重要。比如你要用一批图书构建向量数据库，不同切分粒度会带来不同效果。若整本书只用一个向量，便于识别宏观主题，但会丢失细节；若切到段落甚至句子层面，则更容易检索到具体信息与概念。

语言模型通常受限于一次可处理文本长度，即所谓“上下文窗口”。Chunking 可确保输入不超出该限制，使模型能够通过分段方式处理长文档（如小说）。

在本非评分实验中，你将探索多种切分方法，并观察它们对 RAG 系统的影响！


<div align="center">
  <img src="images/chunking.png" alt="Overview" width="80%">
</div>

<a id='1-1'></a>
### 1.1 导入所需库


In [1]:
from typing import List
import requests
import re
import weaviate
from weaviate.classes.config import Configure, Property, DataType, Tokenization
from weaviate.util import generate_uuid5
import tqdm
from weaviate.classes.query import Filter


In [ ]:
from utils import (
    generate_with_single_input, 
    suppress_subprocess_output,
    kill_processes_on_ports
)

# 在导入 flask_app 之前清理相关端口进程
# 警告：重复运行此单元可能会结束当前内核
kill_processes_on_ports([5000, 8080, 8097, 50050, 50051])

import flask_app

Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 30] Read-only file system: '.models/models--BAAI--bge-base-en-v1.5/.no_exist/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/adapter_config.json'


 * Serving Flask app 'flask_app'
 * Debug mode: off


<a id='1-2'></a>
### 1.2 下载数据

接下来需要一段足够长的文本来体现切分价值。这里我们取 [Pro Git 书](https://git-scm.com/book/en/v2) 中名为 "What is Git?" 的章节作为示例。

In [3]:
url = "https://raw.githubusercontent.com/progit/progit2/main/book/01-introduction/sections/what-is-git.asc"
source_text = requests.get(url).text

In [4]:
print(source_text[:1000])

[[what_is_git_section]]
=== What is Git?

So, what is Git in a nutshell?
This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.
As you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.
Even though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))

==== Snapshots, Not Differences

The major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data.
Conceptually, most other systems store information as a list of file-based changes.
These other systems (CVS, Subversion, Perforce, and so o

In [5]:
print(f"There are about {len(source_text.split())} words in this chapter. Depending on how your LLM tokenizes words, you'd expect roughly {round(len(source_text.split())*1.3)} tokens.")

There are about 1403 words in this chapter. Depending on how your LLM tokenizes words, you'd expect roughly 1824 tokens.


<a id='2'></a>
## 2 - 固定长度切分
---
固定长度切分是指把文本切成大小一致的片段。例如可将文章切成每段 100 个词，或每段 200 个字符。这种方法常见且易用。

其原理是按固定单位数拆分文本，单位可以是词、字符，甚至 token。每个片段的单位数在上限范围内一致，并可选设置片段重叠。


<div align="center">
  <img src="images/fixed_size.png" alt="Fixed Size Chunking" width="80%">
</div>

<a id='2-1'></a>
### 2.1 切分代码示例

下面看一个固定长度切分实现示例。实现方式有很多种，以下只是其中一种。

In [ ]:
def get_chunks_fixed_size(text: str, chunk_size: int) -> List[str]:
    """
    将文本按固定大小切分为多个 chunk。

    参数:
        text (str): 输入文本。
        chunk_size (int): 每个 chunk 的最大词数。

    返回:
        List[str]: 文本 chunk 列表，每个 chunk 最多包含 chunk_size 个词。
    """
    # 将输入文本拆分为词列表
    text_words = text.split()
    
    # 初始化用于存放 chunk 的列表
    chunks = []
    
    # 以 chunk_size 为步长遍历词索引
    for i in range(0, len(text_words), chunk_size):
        # 取从 i 到 i + chunk_size 的子列表
        chunk_words = text_words[i: i + chunk_size]
        
        # 将词用空格拼接成字符串
        chunk = " ".join(chunk_words)
        
        # 将该 chunk 加入结果列表
        chunks.append(chunk)
    
    # 返回全部 chunk
    return chunks

In [7]:
fixed_size_chunks = get_chunks_fixed_size(source_text, chunk_size = 100)

In [8]:
print(len(fixed_size_chunks))

15


In [9]:
fixed_size_chunks[0:3]

["[[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool. Even though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in",
 'a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce))) ==== Snapshots, Not Differences The major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data. Conceptually, most other systems store information as a list of file-based changes. These other systems (CVS, Subversion, Perforce, and s

<a id='2-2'></a>
### 2.2 带重叠切分

下面把代码改造为支持重叠，这样相邻 chunk 会共享一部分 token。


<div align="center">
  <img src="images/overlap.png" alt="带重叠切分" width="80%">
</div>

In [ ]:
def get_chunks_fixed_size_with_overlap(text: str, chunk_size: int, overlap_fraction: float) -> List[str]:
    """
    将文本按固定大小切分，并在相邻 chunk 之间保留指定重叠比例。

    参数:
    - text (str): 输入文本。
    - chunk_size (int): 每个 chunk 的词数。
    - overlap_fraction (float): 邻接 chunk 之间的重叠比例。
      例如 overlap_fraction=0.2 表示重叠 20% 的 chunk 大小。

    返回:
    - List[str]: chunk 列表（字符串），相邻 chunk 可能存在重叠。
    """

    # 将文本拆分为词列表
    text_words = text.split()
    
    # 计算相邻 chunk 之间的重叠词数
    overlap_int = int(chunk_size * overlap_fraction)
    
    # 初始化结果列表
    chunks = []
    
    # 以 chunk_size 为步长遍历并构造 chunk
    for i in range(0, len(text_words), chunk_size):
        # 计算当前 chunk 的起止范围，并考虑与前一块的重叠
        chunk_words = text_words[max(i - overlap_int, 0): i + chunk_size]
        
        # 拼接为字符串 chunk
        chunk = " ".join(chunk_words)
        
        # 加入结果列表
        chunks.append(chunk)
    
    # 返回 chunk 列表
    return chunks

In [ ]:
for chosen_size in [5, 25, 100]:
    chunks = get_chunks_fixed_size_with_overlap(source_text, chosen_size, overlap_fraction=0.2)
    # 打印结果到屏幕
    print(f"\nSize {chosen_size} - {len(chunks)} chunks returned.")
    for i in range(3):
        print(f"Chunk {i+1}: {chunks[i]}")


Size 5 - 281 chunks returned.
Chunk 1: [[what_is_git_section]] === What is Git?
Chunk 2: Git? So, what is Git in
Chunk 3: in a nutshell? This is an

Size 25 - 57 chunks returned.
Chunk 1: [[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git
Chunk 2: if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to
Chunk 3: you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid

Size 100 - 15 chunks returned.
Chunk 1: [[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to clear your mind of

请注意：较小的文本块虽然细节丰富，但可能**信息不足以支持有效检索**。相对地，**较大的文本块通常包含更多信息，更接近常见段落长度**。但当块继续增大时，**其向量表示会逐渐变得更泛化**，最终可能不再适合信息检索。

<a id='3'></a>
## 3 - 可变长度切分 - 递归字符分割

---
现在来看可变长度切分。与固定长度切分不同，这里每个 chunk 的长度是“结果”，而不是“起点”。可变长度切分通常依赖某些分隔标记来切分文本，这些标记可以是句子/段落边界，也可以是 Markdown 标题等结构化元素。

<div align="center">
  <img src="images/recursive.png" alt="Recursive Character Splitting" width="80%">
</div>

<a id='3-1'></a>
### 3.1 可变长度切分方法伪代码

最简单的方法之一是按段落分割（`\n\n`）。

In [ ]:
# 按段落切分文本
def get_chunks_by_paragraph(source_text: str) -> List[str]:
    return source_text.split("\n\n")

另一种方式是在当前文本结构下按章节分割。观察原文可发现，章节由 `\n==` 标记分隔。

In [ ]:
# 按 Asciidoc 章节标记切分文本
def get_chunks_by_asciidoc_sections(source_text: str) -> List[str]:
    return source_text.split("\n==")

In [ ]:
for marker in ["\n\n", "\n=="]:
    chunks = source_text.split(marker)
    # 打印结果到屏幕
    print(f"\nUsing the marker: {repr(marker)} - {len(chunks)} chunks returned.")
    for i in range(3):
        print(f"Chunk {i+1}: {repr(chunks[i])}")


Using the marker: '\n\n' - 31 chunks returned.
Chunk 1: '[[what_is_git_section]]\n=== What is Git?'
Chunk 2: "So, what is Git in a nutshell?\nThis is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.\nAs you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.\nEven though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))"
Chunk 3: '==== Snapshots, Not Differences'

Using the marker: '\n==' - 7 chunks returned.
Chunk 1: '[[what_is_git_section]]'
Chunk 2: "= What is Git?\n\nSo, what is Git in a nutshell?\nThis is an important section to absorb,

基于简单分隔符切分的一个常见问题是：**标题可能会被单独切成一个 chunk**，这通常不理想。实践中可采用混合策略，把较短 chunk（如标题）拼接到后续正文 chunk，让标题与其语义相关内容保持在一起。下面继续探索这种方法。

<a id='3-2'></a>
### 3.2 混合固定与可变长度切分

你可以结合固定长度与可变长度切分，利用两者优势。例如先用可变长度切分器按段落分隔，再加上固定长度约束：若某块过小则与后一块合并；若某块过大则在中间或块内其他标记位置继续拆分。

In [ ]:
def mixed_chunking(source_text):
    """
    混合使用固定长度与可变长度策略对文本进行切分。
    先按 Asciidoc 标记切分，再根据长度做后处理：
    较小 chunk 与后续 chunk 合并，较大 chunk 可在中间或标记处继续拆分。

    参数:
    - source_text (str): 待切分文本。

    返回:
    - list: 文本 chunk 列表。
    """

    # 按 Asciidoc 标记初步切分
    chunks = source_text.split("\n==")

    # 切分后处理逻辑
    new_chunks = []
    chunk_buffer = ""
    min_length = 25

    for chunk in chunks:
        new_buffer = chunk_buffer + chunk  # 生成新的缓冲内容
        new_buffer_words = new_buffer.split(" ")  # 拆分为词
        if len(new_buffer_words) < min_length:  # 判断缓冲内容是否过短
            chunk_buffer = new_buffer  # 留到下一块继续拼接
        else:
            new_chunks.append(new_buffer)  # 写入结果
            chunk_buffer = ""

    if len(chunk_buffer) > 0:
        new_chunks.append(chunk_buffer)  # 如有剩余，追加最后一块

    return new_chunks

In [16]:
mixed_chunks = mixed_chunking(source_text)
for i in range(3):
    print(f"Chunk {i+1}: {repr(mixed_chunks[i])}")

Chunk 1: "[[what_is_git_section]]= What is Git?\n\nSo, what is Git in a nutshell?\nThis is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.\nAs you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.\nEven though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))\n"
Chunk 2: "== Snapshots, Not Differences\n\nThe major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data.\nConceptually, most other systems store information as a list of file-based changes.\nThese other systems (CVS, Subv

这种策略可以在利用语法标记（如标题）确定边界的同时，避免 chunk 过小。完成单篇文本的切分策略分析后，下面看看它们在更大规模文本集合上的表现。

<a id='4'></a>
## 4 - 在真实数据上切分

---
本节及下一节将给出更完整的实践示例。你将对 [Pro Git 书](https://git-scm.com/book/en/v2) 的多个章节应用不同切分方法，并比较它们在搜索任务中的效果。


<a id='4-1'></a>
### 4.1 获取数据

下面获取整本 14 章内容。

In [ ]:
def get_book_text_objects():
    # 数据源位置
    text_objs = list()
    api_base_url = 'https://api.github.com/repos/progit/progit2/contents/book'  # 图书基础 URL
    chapter_urls = ['/01-introduction/sections', '/02-git-basics/sections']  # 章节 URL 列表

    # 遍历图书章节
    for chapter_url in chapter_urls:
        response = requests.get(api_base_url + chapter_url)  # 获取该章节下各 section 文件的 JSON 信息

        # 遍历内部文件（sections）
        for file_info in response.json():
            if file_info['type'] == 'file':  # 仅处理文件（忽略目录）
                file_response = requests.get(file_info['download_url'])

                # 构建包含元数据的对象
                chapter_title = file_info['download_url'].split('/')[-3]
                filename = file_info['download_url'].split('/')[-1]
                text_obj = {
                    "body": file_response.text,
                    "chapter_title": chapter_title,
                    "filename": filename
                }
                text_objs.append(text_obj)
    return text_objs

In [ ]:
# 该函数会生成包含 14 个元素的列表，每个元素对应一个章节
book_text_objs = get_book_text_objects()

In [19]:
print(book_text_objs[0].keys())

dict_keys(['body', 'chapter_title', 'filename'])


<a id='4-2'></a>
### 4.2 章节切分

以下切分方法将应用到每个章节：

- **固定长度切分（20% 重叠）：**
  - 每块 25 词
  - 每块 100 词

- 使用段落标记的 **可变长度切分**

- 使用段落标记且最小长度为 25 词的 **混合策略切分**

此外，每个 chunk 还会附加元数据，包括文件名、章节名和 chunk 编号。

In [ ]:
def build_chunk_objs(book_text_obj, chunks):
    """
    根据给定章节对象及其 chunks 构建 chunk 对象列表。

    参数:
        book_text_obj (dict): 章节元数据字典，包含 chapter_title 和 filename。
        chunks (list): 该章节对应的 chunk 列表。

    返回:
        list: 由字典组成的列表，每个字典包含 chapter_title、filename、chunk、chunk_index。
    """
    chunk_objs = list()  # 初始化 chunk 对象列表
    
    # 带索引遍历所有 chunk
    for i, c in enumerate(chunks):
        # 为每个 chunk 构建对象字典
        chunk_obj = {
            "chapter_title": book_text_obj["chapter_title"],  # 来自章节对象的章节名
            "filename": book_text_obj["filename"],            # 来自章节对象的文件名
            "chunk": c,                                       # 实际文本 chunk
            "chunk_index": i                                  # chunk 在列表中的索引
        }
        # 追加到结果列表
        chunk_objs.append(chunk_obj)

    # 返回 chunk 对象列表
    return chunk_objs

In [ ]:
# 获取多组 chunk（按不同切分策略）
chunk_obj_sets = dict()
for book_text_obj in book_text_objs:
    text = book_text_obj["body"]  # 取当前对象的正文

    # 遍历切分策略：
    for strategy_name, chunks in [
        ["fixed_size_25", get_chunks_fixed_size_with_overlap(text, 25, 0.2)],
        ["fixed_size_100", get_chunks_fixed_size_with_overlap(text, 100, 0.2)],
        ["para_chunks", get_chunks_by_paragraph(text)],
        ["para_chunks_min_25", mixed_chunking(text)]
    ]:
        chunk_objs = build_chunk_objs(book_text_obj, chunks)

        if strategy_name not in chunk_obj_sets.keys():
            chunk_obj_sets[strategy_name] = list()

        chunk_obj_sets[strategy_name] += chunk_objs

In [22]:
print(chunk_obj_sets.keys())

dict_keys(['fixed_size_25', 'fixed_size_100', 'para_chunks', 'para_chunks_min_25'])


In [23]:
chunk_type = 'fixed_size_25' # Change it to check the different chunks!
chunk_obj_sets[chunk_type][0:2]

[{'chapter_title': '01-introduction',
  'filename': 'about-version-control.asc',
  'chunk': '=== About Version Control (((version control))) What is "`version control`", and why should you care? Version control is a system that records changes to a',
  'chunk_index': 0},
 {'chapter_title': '01-introduction',
  'filename': 'about-version-control.asc',
  'chunk': 'that records changes to a file or set of files over time so that you can recall specific versions later. For the examples in this book, you will use software',
  'chunk_index': 1}]

<a id='4-3'></a>
### 4.3 将切分结果加载到向量数据库

本节重点是把 chunk 加载到向量数据库。下面给出了创建并写入向量数据库的大致流程。不过为节省时间，本实验使用了预加载的 collection。如果你还没完成 Weaviate API 非评分实验，建议先完成，以便更好理解该流程。

In [ ]:
import os

# 清理可能占用 Weaviate 端口的进程
kill_processes_on_ports([8080, 8079, 50050, 50051])

# 加载 client
collection_base = os.getenv('COLLECTION_M3', './')
persistence_path = os.path.join(collection_base, 'ungraded_lab_2')

with suppress_subprocess_output():
    try:
        client = weaviate.connect_to_embedded(
            persistence_data_path=persistence_path,
            environment_variables={
                "ENABLE_API_BASED_MODULES": "true", # 启用 API 模块
                "ENABLE_MODULES": 'text2vec-transformers', # 使用 transformer 向量化模型
                "TRANSFORMERS_INFERENCE_API":"http://127.0.0.1:5000/", # Weaviate 用于向量化的推理端点
            }
        )
    except Exception as e:
        print(f"Failed to connect to embedded Weaviate: {e}")
        print("Falling back to local Weaviate connection...")
        try:
            client = weaviate.connect_to_local(port=8079, grpc_port=50050)
        except Exception as e2:
            print(f"Failed to connect to local Weaviate: {e2}")
            print("Trying alternative ports...")
            client = weaviate.connect_to_local(port=8080, grpc_port=50051)

In [ ]:
# 创建 collection
if not client.collections.exists("chunking_example"):
    collection = client.collections.create(
            name='chunking_example',

            vectorizer_config=[Configure.NamedVectors.text2vec_transformers(
                    name="vector", # 在 collection 中访问对象向量时使用的名称
                    #source_properties=['chunk'], # 指定用于向量化的属性，会先拼接再向量化
                    vectorize_collection_name = False, # 不对 collection 名称做向量化
                                                       # 若为 True，会拼接到待向量化文本前部
                    inference_url="http://127.0.0.1:5000", # 使用 API 向量化器时需要提供调用端点
                                                           # 该端点由本实验 Flask 应用提供
                )],

            properties=[  # 定义 properties
            Property(name="chunk",data_type= DataType.TEXT),
            Property(name="chapter_title", data_type=DataType.TEXT),
            Property(name="filename",data_type=DataType.TEXT),
            Property(name="chunking_strategy",data_type=DataType.TEXT, tokenization = Tokenization.FIELD), # Tokenization.FIELD 表示整体词项作为一个 token
            Property(name="chunk_index",data_type=DataType.INT),

        ]
        )
else:
    collection = client.collections.get("chunking_example")

<div class="alert alert-block alert-warning">  
<b>注意</b> <a class="tocSkip"></a><br> 


```python
# 向 collection 写入元素 - 此单元不应运行，因为该 collection 已提前向量化。 
if len(collection) == 0:
    with collection.batch.fixed_size(batch_size=1, concurrent_requests=20) as batch:
        for chunking_strategy, chunk_objects in tqdm.tqdm(chunk_obj_sets.items()):
            for chunk_obj in chunk_objects:
                chunk_obj["chunking_strategy"] = chunking_strategy
                batch.add_object(
                    properties=chunk_obj,
                    uuid=generate_uuid5(chunk_obj)
                )
```

</div>

In [ ]:
print(f"Total count: {collection.aggregate.over_all().total_count}")
for chunking_strategy in chunk_obj_sets.keys():
    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy) # 按切分策略过滤
    count = collection.aggregate.over_all(filters = where_filter).total_count # 在过滤条件下聚合统计
    print(f"Object count for {chunking_strategy}: {count}")

Total count: 1487
Object count for fixed_size_25: 672
Object count for fixed_size_100: 173
Object count for para_chunks: 549
Object count for para_chunks_min_25: 93


<a id='5'></a>
## 5 - 搜索 
---
本节将使用不同 chunk 粒度进行语义搜索，以观察粒度对信息检索效果的影响。

In [27]:
search_string = "history of git"  # Or "available git remote commands"

for chunking_strategy in chunk_obj_sets.keys():
    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)
    response = collection.query.near_text(search_string, filters = where_filter, limit = 2)
    print(f"RETRIEVED OBJECTS FOR CHUNKING STRATEGY {chunking_strategy.upper()}:\n")
    for i, obj in enumerate(response.objects):
        print(f"===== Object {i} =====")
        print(f"{obj.properties['chunk']}")
        print()

RETRIEVED OBJECTS FOR CHUNKING STRATEGY FIXED_SIZE_25:

===== Object 0 =====
=== A Short History of Git As with many great things in life, Git began with a bit of creative destruction and fiery controversy. The

===== Object 1 =====
kernel efficiently (speed and data size) Since its birth in 2005, Git has evolved and matured to be easy to use and yet retain these initial qualities. It's amazingly fast,

RETRIEVED OBJECTS FOR CHUNKING STRATEGY FIXED_SIZE_100:

===== Object 0 =====
=== A Short History of Git As with many great things in life, Git began with a bit of creative destruction and fiery controversy. The Linux kernel is an open source software project of fairly large scope.(((Linux))) During the early years of the Linux kernel maintenance (1991–2002), changes to the software were passed around as patches and archived files. In 2002, the Linux kernel project began using a proprietary DVCS called BitKeeper.(((BitKeeper))) In 2005, the relationship between the community that develo

在这个示例中，查询是较宽泛的 "history of git"。结果表明较长 chunk 通常表现更好。进一步查看会发现，虽然 25 词 chunk 在语义相似度上可能更接近查询，但它们缺少足够上下文，难以显著提升读者理解；而检索到的段落级 chunk（尤其是最小长度 25 词版本）提供了更完整的信息，更适合帮助读者理解 Git 的历史。

In [28]:
search_string = "how to add the url of a remote repository"  # Or "available git remote commands"

for chunking_strategy in chunk_obj_sets.keys():
    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)
    response = collection.query.near_text(search_string, filters = where_filter, limit = 2)
    print(f"RETRIEVED OBJECTS FOR CHUNKING STRATEGY {chunking_strategy.upper()}:\n")
    for i, obj in enumerate(response.objects):
        print(f"===== Object {i} =====")
        print(f"{obj.properties['chunk']}")
        print()

RETRIEVED OBJECTS FOR CHUNKING STRATEGY FIXED_SIZE_25:

===== Object 0 =====
remote))) To add a new remote Git repository as a shortname you can reference easily, run `git remote add <shortname> <url>`: [source,console] ---- $ git remote origin $ git remote

===== Object 1 =====
manage your remote repositories. Remote repositories are versions of your project that are hosted on the Internet or network somewhere. You can have several of them, each of which generally

RETRIEVED OBJECTS FOR CHUNKING STRATEGY FIXED_SIZE_100:

===== Object 0 =====
adds the `origin` remote for you. Here's how to add a new remote explicitly.(((git commands, remote))) To add a new remote Git repository as a shortname you can reference easily, run `git remote add <shortname> <url>`: [source,console] ---- $ git remote origin $ git remote add pb https://github.com/paulboone/ticgit $ git remote -v origin https://github.com/schacon/ticgit (fetch) origin https://github.com/schacon/ticgit (push) pb https://github.com

在这个示例中，查询更具体，例如用户想了解如何添加远程仓库 URL。与前一个场景不同，25 词 chunk 在这里更有帮助。因为问题足够具体，Weaviate 能更精准定位最相关片段，即如何添加远程仓库（`git remote add <shortname> <url>`）。

虽然其他结果集合也包含相关信息，但还要考虑结果如何被使用和展示。更长结果通常需要用户投入更多认知成本来提取关键信息。

<a id='6'></a>
## 6 - 在 RAG 系统中集成
---
现在你已经熟悉了 chunking，并且已有可用 collection。下面看看不同 chunk 粒度如何影响文本生成效果。先使用一个简单提示词。

In [29]:
PROMPT = "Using this information and only this information, please explain {search_string} in a few short points.\nContext: {context}"

In [ ]:
# 设置每种策略的检索块数，以平衡不同 chunk 粒度

n_chunks_by_strat = dict()

# 对更短 chunk 取更多结果
n_chunks_by_strat['fixed_size_25'] = 8
n_chunks_by_strat['para_chunks'] = 8

# 对更长 chunk 取更少结果
n_chunks_by_strat['fixed_size_100'] = 2
n_chunks_by_strat['para_chunks_min_25'] = 2

# 执行检索增强生成
search_string = "history of git"  # Or "available git remote commands"

for chunking_strategy in chunk_obj_sets.keys():
    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)
    response = collection.query.near_text(search_string, filters = where_filter, limit = n_chunks_by_strat[chunking_strategy])
    context_string = ""
    for obj in response.objects:
        context_string += obj.properties['chunk'] + '\n'
    prompt = PROMPT.format(search_string = search_string, context = context_string)
    response = generate_with_single_input(prompt, role = 'assistant')
    print(f"Search string: {search_string}")
    print(f"Chunking Strategy: {chunking_strategy}:")
    print(f"Response:\n\t{response['content']}")
    print()

Search string: history of git
Chunking Strategy: fixed_size_25:
Response:
	Based strictly on the provided text, here is the short history of Git:

*   **Origins and Philosophy**: Git began in 2005 rooted in "creative destruction" and "fiery controversy." Its primary design goals were to be efficient regarding speed and data size while eventually maturing into an easy-to-use system.
*   **Early Commitments**: The software saw early developments by authors such as Scott Chacon, with notable commits recorded in March 2008 (e.g., initial commits and file modifications like `simplegit.rb`). Other historical snapshots were committed by Junio Hamano in October 2008.
*   **Historical Analysis**: To trace this history, users primarily utilize the `git log` command to view existing commit histories, though the text notes that this output can be "quite wordy" and is often filtered for comprehensiveness.

Search string: history of git
Chunking Strategy: fixed_size_100:
Response:
	Based on the prov

In [ ]:
# 别忘了关闭 client！
client.close()

恭喜你！你已完成关于 Chunking 的非评分实验！继续加油！